# LeetCode 1684 — Count the Number of Consistent Strings bằng Python

**Link đề:** https://leetcode.com/problems/count-the-number-of-consistent-strings/
**Độ khó:** Easy · **Chủ đề:** Array, Hash Table, String, Bit Manipulation, Counting

```python
class Solution:
    def countConsistentStrings(self, allowed: str, words: List[str]) -> int:
        ...
```

## Mục tiêu học
1. Hiểu đúng khái niệm **chuỗi nhất quán (consistent)**: *mọi* ký tự của từ đều nằm trong `allowed`.
2. Nắm các công cụ Python: duyệt chuỗi, toán tử `in`, `set`, `all()`, generator expression, `sum()` trên giá trị bool, phép tập con `<=`.
3. Đi từ cách trực giác (tra `in` trên chuỗi) → dùng `set` để tra nhanh → so sánh tập con → **bitmask 26 bit**.
4. Trace từng từ, thấy rõ thời điểm "phát hiện ký tự lạ thì dừng".
5. Tự viết lại lời giải, nhận diện pattern "kiểm tra mọi phần tử thuộc một tập cho phép".

## 1. Đề bài và ví dụ

Cho chuỗi `allowed` gồm các ký tự **khác nhau** và mảng chuỗi `words`. Một chuỗi gọi là **consistent** nếu **tất cả** ký tự của nó đều xuất hiện trong `allowed`. Trả về **số lượng** chuỗi consistent trong `words`.

| `allowed` | `words` | Kết quả | Giải thích |
|---|---|:---:|---|
| `"ab"` | `["ad","bd","aaab","baa","badab"]` | `2` | chỉ `"aaab"`, `"baa"` không chứa ký tự lạ |
| `"abc"` | `["a","b","c","ab","ac","bc","abc"]` | `7` | mọi từ đều chỉ dùng a, b, c |
| `"cad"` | `["cc","acd","b","ba","bac","bad","ac","d"]` | `4` | `"cc"`, `"acd"`, `"ac"`, `"d"` |

**Constraints:**
- `1 <= len(words) <= 10^4`
- `1 <= len(allowed) <= 26`
- `1 <= len(words[i]) <= 10`
- Các ký tự trong `allowed` khác nhau đôi một.
- `words[i]` và `allowed` chỉ gồm chữ thường tiếng Anh.

Câu hỏi then chốt: với mỗi từ, làm sao kiểm tra **nhanh** "có ký tự nào *không* thuộc `allowed` không?"

### 1.1. Cách hiểu đúng: "mọi ký tự", không phải "có ký tự"

`"ad"` với `allowed = "ab"`: chữ `a` hợp lệ nhưng chữ `d` thì không → **không** consistent. Chỉ cần **một** ký tự lạ là từ bị loại. Lặp lại ký tự không sao: `"aaab"` vẫn consistent. Ta kiểm tra bằng tay trước khi code.

In [1]:
from typing import List

allowed = "ab"
words = ["ad", "bd", "aaab", "baa", "badab"]
for w in words:
    la = [ch for ch in w if ch not in allowed]   # các ký tự "lạ"
    print(f"{w:>6} -> ký tự lạ: {la} -> consistent = {len(la) == 0}")

    ad -> ký tự lạ: ['d'] -> consistent = False
    bd -> ký tự lạ: ['d'] -> consistent = False
  aaab -> ký tự lạ: [] -> consistent = True
   baa -> ký tự lạ: [] -> consistent = True
 badab -> ký tự lạ: ['d'] -> consistent = False


## 2. Nền tảng Python cần biết

### 2.1. Duyệt từng ký tự của chuỗi

Chuỗi (`str`) là một dãy ký tự. `for ch in w:` lần lượt lấy từng ký tự từ trái sang phải. `len(w)` là số ký tự.

In [2]:
w = "baa"
print("len =", len(w))
for i, ch in enumerate(w):
    print("vị trí", i, "ký tự", ch)

len = 3
vị trí 0 ký tự b
vị trí 1 ký tự a
vị trí 2 ký tự a


### 2.2. Toán tử `in` trên chuỗi

`ch in allowed` trả `True` nếu ký tự `ch` có trong chuỗi `allowed`. Với chuỗi, Python phải **quét** từ đầu đến khi gặp → tốn tối đa `len(allowed)` bước (≤ 26 ở bài này). `not in` là phủ định.

In [3]:
allowed = "cad"
print("'a' in 'cad' ->", 'a' in allowed)
print("'b' in 'cad' ->", 'b' in allowed)
print("'b' not in 'cad' ->", 'b' not in allowed)

'a' in 'cad' -> True
'b' in 'cad' -> False
'b' not in 'cad' -> True


### 2.3. `set` — tập hợp, tra cứu trung bình `O(1)`

`set(allowed)` tạo tập các ký tự. Tập không có phần tử trùng và không có thứ tự. Phép `ch in s` trên `set` dùng **băm (hash)** nên trung bình `O(1)`, không cần quét. Ở bài này `allowed` ≤ 26 nên khác biệt nhỏ, nhưng thói quen dùng `set` cho "tra có/không" rất quan trọng.

In [4]:
allowed_set = set("cad")
print(allowed_set)              # thứ tự in có thể khác nhau
print('c' in allowed_set, 'z' in allowed_set)
print(set("aaab"))              # trùng bị gộp -> {'a', 'b'}

{'c', 'd', 'a'}
True False
{'b', 'a'}


### 2.4. `all()` và generator expression

`all(iterable)` trả `True` nếu **mọi** phần tử đều đúng (và trả `True` với dãy rỗng). Nó **dừng sớm** ngay khi gặp phần tử sai.

`(ch in allowed_set for ch in w)` là *generator expression*: sinh lần lượt từng giá trị bool, không tạo list trung gian. Viết gọn: `all(ch in allowed_set for ch in w)`.

In [5]:
allowed_set = set("ab")
print(all(ch in allowed_set for ch in "aaab"))   # True
print(all(ch in allowed_set for ch in "bd"))     # False (dừng ở 'd')
print(all([]))                                   # True: không có phần tử sai nào

def noisy(ch):
    print("  kiểm tra", ch)
    return ch in allowed_set
print("kết quả:", all(noisy(ch) for ch in "abzab"))   # chỉ kiểm tra đến 'z'

True
False
True
  kiểm tra a
  kiểm tra b
  kiểm tra z
kết quả: False


### 2.5. `sum()` trên giá trị bool

Trong Python, `True` hoạt động như `1` và `False` như `0`. Vì vậy `sum(điều_kiện for ...)` **đếm** số lần điều kiện đúng.

In [6]:
print(True + True + False)          # 2
flags = [True, False, True, True]
print(sum(flags))                   # 3
print(sum(x > 2 for x in [1, 5, 3, 0]))   # đếm số > 2 -> 2

2
3
2


### 2.6. Phép tập con `<=` giữa hai `set`

`A <= B` (hoặc `A.issubset(B)`) đúng khi **mọi phần tử của A đều thuộc B**. Đây chính xác là định nghĩa "consistent" nếu A = tập ký tự của từ, B = tập `allowed`.

In [7]:
B_ = set("ab")
print(set("aaab") <= B_)     # {'a','b'} ⊆ {'a','b'} -> True
print(set("bd") <= B_)       # 'd' không thuộc -> False
print(set("baa").issubset(B_))

True
False
True


### 2.7. Bitmask: dùng 26 bit thay cho tập chữ cái

Chỉ có 26 chữ thường → có thể biểu diễn một tập chữ bằng **một số nguyên**, bit thứ `i` bật nếu chữ thứ `i` (a=0, b=1, …) có trong tập.

- `ord(ch) - ord('a')` đổi chữ thành chỉ số 0..25.
- `1 << i` là số chỉ có bit `i` bật (bằng `2**i`).
- `mask | (1 << i)` bật thêm bit `i` (phép OR).
- `mask & bit` khác 0 nếu bit đó đang bật (phép AND).

In [8]:
for ch in "abd":
    i = ord(ch) - ord('a')
    print(ch, "-> chỉ số", i, "-> 1 << i =", 1 << i, "=", bin(1 << i))

mask = 0
for ch in "ab":
    mask |= 1 << (ord(ch) - ord('a'))
print("mask của 'ab' =", mask, bin(mask))
print("'b' có trong mask?", (mask & (1 << 1)) != 0)
print("'d' có trong mask?", (mask & (1 << 3)) != 0)

a -> chỉ số 0 -> 1 << i = 1 = 0b1
b -> chỉ số 1 -> 1 << i = 2 = 0b10
d -> chỉ số 3 -> 1 << i = 8 = 0b1000
mask của 'ab' = 3 0b11
'b' có trong mask? True
'd' có trong mask? False


**Kiểm tra tập con bằng bit:** từ `w` có mask `wm`, allowed có mask `am`. `w` consistent ⇔ `wm` không có bit nào nằm ngoài `am` ⇔ `wm & ~am == 0`, tương đương `wm | am == am`.

In [9]:
def to_mask(s):
    m = 0
    for ch in s:
        m |= 1 << (ord(ch) - ord('a'))
    return m

am = to_mask("ab")
for w in ["aaab", "bd"]:
    wm = to_mask(w)
    print(f"{w:>5}: wm={bin(wm):>8}, am={bin(am)}, wm | am == am -> {wm | am == am}")

 aaab: wm=    0b11, am=0b11, wm | am == am -> True
   bd: wm=  0b1010, am=0b11, wm | am == am -> False


## 3. Cách trực giác — tra `in` trực tiếp trên chuỗi `allowed`

Ý tưởng tự nhiên nhất: duyệt từng từ, duyệt từng ký tự; nếu thấy ký tự không có trong `allowed` thì đánh dấu "hỏng" và bỏ qua phần còn lại. Cuối mỗi từ, nếu không hỏng thì `count += 1`.

In [10]:
def count_consistent_naive(allowed: str, words: List[str]) -> int:
    count = 0
    for w in words:
        ok = True
        for ch in w:
            if ch not in allowed:     # quét chuỗi allowed (<= 26 ký tự)
                ok = False
                break                 # đã hỏng, không cần xem tiếp
        if ok:
            count += 1
    return count

print(count_consistent_naive("ab", ["ad","bd","aaab","baa","badab"]))      # 2
print(count_consistent_naive("abc", ["a","b","c","ab","ac","bc","abc"]))  # 7
print(count_consistent_naive("cad", ["cc","acd","b","ba","bac","bad","ac","d"]))  # 4

2
7
4


### Chi phí của cách trực giác

Gọi `N = len(words)`, `L` = độ dài tối đa một từ (≤ 10), `A = len(allowed)` (≤ 26). Mỗi phép `ch in allowed` (trên chuỗi) tốn `O(A)` → tổng `O(N · L · A)`. Với constraints này là ≤ 10^4 · 10 · 26 ≈ 2,6 triệu phép so sánh — **vẫn AC** (gợi ý của đề cũng nói brute force đủ nhanh).

## 4. Nút thắt và hướng cải tiến

Công việc lặp lại: mỗi lần hỏi "`ch` có trong `allowed` không?" ta lại **quét lại** `allowed`. Nhưng `allowed` không đổi! → **Tiền xử lý một lần** thành cấu trúc tra nhanh:
- `set(allowed)` → mỗi lần tra trung bình `O(1)`.
- hoặc mảng 26 phần tử `bool` → tra `O(1)` tuyệt đối.
- hoặc bitmask → tra/so sánh bằng vài phép bit.

Đây là mẫu tư duy chung: **dữ liệu cố định bị tra cứu nhiều lần → tiền xử lý thành bảng tra.**

In [11]:
# Đếm số lần phép "in" phải so sánh ký tự trong cách trực giác
def count_compare_steps(allowed, words):
    steps = 0
    for w in words:
        for ch in w:
            # vị trí ch trong allowed (+1), hoặc len(allowed) nếu không có
            pos = allowed.find(ch)
            steps += (pos + 1) if pos != -1 else len(allowed)
            if pos == -1:
                break
    return steps

print("số bước so sánh (ví dụ 3):", count_compare_steps("cad", ["cc","acd","b","ba","bac","bad","ac","d"]))
print("số lần tra set tương ứng  :", sum(len(w) for w in ["cc","acd","b","ba","bac","bad","ac","d"]), "(tối đa)")

số bước so sánh (ví dụ 3): 26
số lần tra set tương ứng  : 17 (tối đa)


## 5. Lời giải chính — `set` + `all()` + `sum()`

Ta mổ xẻ từng đoạn rồi ghép lại.

**Bước 1 — tiền xử lý:** `allowed_set = set(allowed)`.

**Bước 2 — kiểm tra một từ:** `all(ch in allowed_set for ch in w)` → `True` nếu từ consistent.

**Bước 3 — đếm:** cộng các giá trị bool ở bước 2 cho mọi từ.

In [12]:
allowed_set = set("cad")                                   # bước 1
w = "acd"
print("bước 2 với 'acd':", all(ch in allowed_set for ch in w))
w = "bad"
print("bước 2 với 'bad':", all(ch in allowed_set for ch in w))

bước 2 với 'acd': True
bước 2 với 'bad': False


Ghép lại thành hàm hoàn chỉnh (viết rõ ràng, có vòng `for` ngoài để dễ đọc):

In [13]:
def count_consistent_set(allowed: str, words: List[str]) -> int:
    allowed_set = set(allowed)               # tiền xử lý O(A)
    count = 0
    for w in words:
        if all(ch in allowed_set for ch in w):   # dừng sớm khi gặp ký tự lạ
            count += 1
    return count

print(count_consistent_set("ab", ["ad","bd","aaab","baa","badab"]))       # 2
print(count_consistent_set("abc", ["a","b","c","ab","ac","bc","abc"]))   # 7
print(count_consistent_set("cad", ["cc","acd","b","ba","bac","bad","ac","d"]))  # 4

2
7
4


## 6. TRACE từng bước

Ví dụ 3: `allowed = "cad"`, `words = ["cc","acd","b","ba","bac","bad","ac","d"]`.

| Từ | Ký tự được kiểm tra | Gặp ký tự lạ? | consistent | count sau |
|---|---|---|:---:|:---:|
| `cc` | c ✓, c ✓ | không | ✓ | 1 |
| `acd` | a ✓, c ✓, d ✓ | không | ✓ | 2 |
| `b` | b ✗ | `b` | ✗ | 2 |
| `ba` | b ✗ (dừng) | `b` | ✗ | 2 |
| `bac` | b ✗ (dừng) | `b` | ✗ | 2 |
| `bad` | b ✗ (dừng) | `b` | ✗ | 2 |
| `ac` | a ✓, c ✓ | không | ✓ | 3 |
| `d` | d ✓ | không | ✓ | 4 |

Chú ý: với `"ba"`, ta **không** cần kiểm tra `a` vì `b` đã làm hỏng từ.

In [14]:
def trace_count_consistent(allowed: str, words: List[str]) -> int:
    allowed_set = set(allowed)
    print("allowed_set =", sorted(allowed_set))
    count = 0
    for w in words:
        checked = []
        bad = None
        for ch in w:
            if ch in allowed_set:
                checked.append(ch + "✓")
            else:
                checked.append(ch + "✗")
                bad = ch
                break
        if bad is None:
            count += 1
        print(f"{w:>5} | kiểm tra: {' '.join(checked):<12} | lạ: {str(bad):<4} | count = {count}")
    return count

trace_count_consistent("cad", ["cc","acd","b","ba","bac","bad","ac","d"])

allowed_set = ['a', 'c', 'd']
   cc | kiểm tra: c✓ c✓        | lạ: None | count = 1
  acd | kiểm tra: a✓ c✓ d✓     | lạ: None | count = 2
    b | kiểm tra: b✗           | lạ: b    | count = 2
   ba | kiểm tra: b✗           | lạ: b    | count = 2
  bac | kiểm tra: b✗           | lạ: b    | count = 2
  bad | kiểm tra: b✗           | lạ: b    | count = 2
   ac | kiểm tra: a✓ c✓        | lạ: None | count = 3
    d | kiểm tra: d✓           | lạ: None | count = 4


4

## 7. Các lời giải khác (từ cơ bản đến sáng tạo)

### 7.1. Cách A — Phép tập con `set(w) <= allowed_set`

Mỗi từ đổi thành tập ký tự rồi hỏi "có là tập con không". Gọn, rất "Pythonic". Nhược điểm nhỏ: luôn dựng `set(w)` đủ cả từ (không dừng sớm), nhưng từ ≤ 10 ký tự nên không đáng kể.

In [15]:
def count_consistent_subset(allowed: str, words: List[str]) -> int:
    allowed_set = set(allowed)
    return sum(set(w) <= allowed_set for w in words)

print(count_consistent_subset("ab", ["ad","bd","aaab","baa","badab"]))       # 2
print(count_consistent_subset("cad", ["cc","acd","b","ba","bac","bad","ac","d"]))  # 4

2
4


### 7.2. Cách B — Mảng đánh dấu 26 ô

Không dùng hash: tạo `ok = [False]*26`, bật `ok[ord(ch)-97] = True` cho chữ trong `allowed`. Tra cứu là truy cập mảng theo chỉ số — `O(1)` tuyệt đối. Đây là cách hay dùng trong C/Java.

In [16]:
def count_consistent_array(allowed: str, words: List[str]) -> int:
    ok = [False] * 26
    for ch in allowed:
        ok[ord(ch) - ord('a')] = True
    count = 0
    for w in words:
        good = True
        for ch in w:
            if not ok[ord(ch) - ord('a')]:
                good = False
                break
        count += good          # True -> +1, False -> +0
    return count

print(count_consistent_array("abc", ["a","b","c","ab","ac","bc","abc"]))   # 7

7


### 7.3. Cách C — Bitmask 26 bit

Biến `allowed` thành số nguyên `am`. Với mỗi từ, dựng `wm` rồi kiểm tra `wm | am == am` (không có bit nào nằm ngoài `am`). Rất nhanh, bộ nhớ chỉ là vài số nguyên.

In [17]:
def count_consistent_bitmask(allowed: str, words: List[str]) -> int:
    am = 0
    for ch in allowed:
        am |= 1 << (ord(ch) - ord('a'))
    count = 0
    for w in words:
        wm = 0
        for ch in w:
            wm |= 1 << (ord(ch) - ord('a'))
        if wm | am == am:          # wm là "tập con" của am
            count += 1
    return count

print(count_consistent_bitmask("cad", ["cc","acd","b","ba","bac","bad","ac","d"]))  # 4

4


### 7.4. Cách D — Đếm ngược: lấy tổng trừ số từ "hỏng"

Đôi khi dễ nghĩ theo chiều phủ định: một từ **hỏng** nếu `any(ch not in allowed_set for ch in w)`. Đáp án = `len(words) - số_từ_hỏng`. `any()` là anh em của `all()`: `True` nếu **có ít nhất một** phần tử đúng.

In [18]:
def count_consistent_complement(allowed: str, words: List[str]) -> int:
    allowed_set = set(allowed)
    bad = sum(any(ch not in allowed_set for ch in w) for w in words)
    return len(words) - bad

print(count_consistent_complement("ab", ["ad","bd","aaab","baa","badab"]))   # 2
print("any([False, True]) =", any([False, True]), "| any([]) =", any([]))

2
any([False, True]) = True | any([]) = False


### 7.5. Bảng so sánh

| Cách | Ý tưởng | Time | Space phụ | Ghi chú |
|---|---|---|---|---|
| Trực giác | `ch in allowed` (chuỗi) | `O(N·L·A)` | `O(1)` | dễ nhất, vẫn AC |
| Set + all | tra `set` | `O(A + N·L)` trung bình | `O(A)` | khuyên dùng |
| Tập con | `set(w) <= allowed_set` | `O(A + N·L)` trung bình | `O(A + L)` | ngắn nhất |
| Mảng 26 ô | `ok[ord(ch)-97]` | `O(A + N·L)` | `O(26)` | không phụ thuộc hash |
| Bitmask | `wm | am == am` | `O(A + N·L)` | `O(1)` | nhanh, đẹp, ít bộ nhớ |

Trong đó `N = len(words)`, `L` = độ dài từ dài nhất, `A = len(allowed)`.

## 8. Code nộp LeetCode

Bản chính (set + all). LeetCode đã `from typing import List` sẵn; trong notebook ta import ở cell đầu.

In [19]:
class Solution:
    def countConsistentStrings(self, allowed: str, words: List[str]) -> int:
        allowed_set = set(allowed)
        return sum(all(ch in allowed_set for ch in w) for w in words)

sol = Solution()
print(sol.countConsistentStrings("ab", ["ad","bd","aaab","baa","badab"]))      # 2
print(sol.countConsistentStrings("abc", ["a","b","c","ab","ac","bc","abc"]))  # 7
print(sol.countConsistentStrings("cad", ["cc","acd","b","ba","bac","bad","ac","d"]))  # 4

2
7
4


Bản bitmask để nộp (nếu muốn luyện thao tác bit):

In [20]:
class SolutionBitmask:
    def countConsistentStrings(self, allowed: str, words: List[str]) -> int:
        am = 0
        for ch in allowed:
            am |= 1 << (ord(ch) - 97)
        ans = 0
        for w in words:
            wm = 0
            for ch in w:
                wm |= 1 << (ord(ch) - 97)
            ans += (wm & ~am) == 0     # không có bit nào ngoài am
        return ans

print(SolutionBitmask().countConsistentStrings("cad", ["cc","acd","b","ba","bac","bad","ac","d"]))  # 4

4


## 9. Vì sao thuật toán đúng?

**Định nghĩa:** `w` consistent ⇔ với **mọi** `ch` trong `w`, `ch ∈ allowed`.

- `all(ch in allowed_set for ch in w)` dịch nguyên văn mệnh đề "với mọi" này. `set(allowed)` chứa đúng các ký tự của `allowed` nên `ch in allowed_set ⇔ ch ∈ allowed`.
- **Dừng sớm vẫn đúng:** khi gặp ký tự lạ, mệnh đề "với mọi" đã sai, xem tiếp không thể làm nó đúng lại.
- **Invariant của vòng đếm:** sau khi xử lý `k` từ đầu, `count` = số từ consistent trong `words[0..k-1]`. Ban đầu `k=0, count=0` đúng; mỗi bước cộng 1 đúng khi từ thứ `k` consistent → giữ invariant; khi `k = N` ta có đáp án.
- **Bitmask:** `wm` có bit `i` bật ⇔ chữ thứ `i` xuất hiện trong `w`. `wm & ~am == 0` ⇔ mọi bit bật của `wm` cũng bật trong `am` ⇔ mọi chữ của `w` thuộc `allowed`.

## 10. Độ phức tạp

Gọi `N = len(words)`, `L` = độ dài tối đa của từ, `A = len(allowed)`, `S = tổng số ký tự của mọi từ` (≤ `N·L`).

- **Time (set):** dựng set `O(A)` + duyệt mọi ký tự `O(S)` → `O(A + S)` trung bình (tra hash trung bình `O(1)`).
- **Space:** `O(A)` cho `allowed_set` (≤ 26 → thực chất `O(1)`).
- **Bitmask / mảng 26 ô:** `O(A + S)` thời gian không cần "trung bình", bộ nhớ `O(1)`.
- **Trực giác:** `O(S · A)` vì mỗi `in` trên chuỗi quét tối đa `A` ký tự.

Ta không thể nhanh hơn `O(S)` trong trường hợp xấu nhất vì phải đọc từng ký tự ít nhất một lần (mọi từ đều có thể consistent).

In [21]:
import time, random, string

random.seed(1)
allowed_big = "".join(random.sample(string.ascii_lowercase, 20))
words_big = ["".join(random.choice(allowed_big) for _ in range(10)) for _ in range(10**4)]

for name, f in [("naive", count_consistent_naive), ("set", count_consistent_set),
                ("subset", count_consistent_subset), ("bitmask", count_consistent_bitmask)]:
    t = time.perf_counter()
    r = f(allowed_big, words_big)
    print(f"{name:>8}: kết quả={r}, thời gian={time.perf_counter() - t:.4f}s")

   naive: kết quả=10000, thời gian=0.0034s
     set: kết quả=10000, thời gian=0.0046s
  subset: kết quả=10000, thời gian=0.0063s
 bitmask: kết quả=10000, thời gian=0.0083s


**Nhận xét thời gian thực tế:** với `A ≤ 26`, phép `ch in allowed` trên chuỗi được cài bằng C nên rất nhanh — cách trực giác có thể còn nhanh hơn trên máy thật! Big-O nói về **xu hướng khi dữ liệu lớn dần**; khi `A` nhỏ cố định, hằng số mới quyết định. Bài học: cả bốn cách đều AC; chọn cách **rõ nghĩa nhất** (set + all), và hiểu vì sao set/bitmask mở rộng tốt hơn khi tập cho phép lớn (ví dụ tập từ khoá hàng nghìn phần tử).

## 11. Bộ test: mẫu đề, biên, phản ví dụ, đối chiếu

Nhóm test:
1. Ba ví dụ của đề.
2. Biên: một từ, một ký tự; `allowed` đủ 26 chữ (mọi từ đều consistent).
3. Phản ví dụ dễ sai: từ có ký tự lạ ở **cuối** (kiểm tra không dừng quá sớm); từ lặp ký tự.
4. Đối chiếu 6 cách trên nhiều input ngẫu nhiên.

In [22]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"PASS {name}: {actual}")

S = Solution()
check("đề 1", S.countConsistentStrings("ab", ["ad","bd","aaab","baa","badab"]), 2)
check("đề 2", S.countConsistentStrings("abc", ["a","b","c","ab","ac","bc","abc"]), 7)
check("đề 3", S.countConsistentStrings("cad", ["cc","acd","b","ba","bac","bad","ac","d"]), 4)
check("1 từ hợp lệ", S.countConsistentStrings("a", ["a"]), 1)
check("1 từ không hợp lệ", S.countConsistentStrings("a", ["b"]), 0)
check("đủ 26 chữ", S.countConsistentStrings("abcdefghijklmnopqrstuvwxyz", ["xyz", "hello", "q"]), 3)
check("ký tự lạ ở cuối", S.countConsistentStrings("ab", ["abababz"]), 0)
check("lặp ký tự", S.countConsistentStrings("z", ["zzzzzzzzzz", "zzzzzzzzza"]), 1)

PASS đề 1: 2
PASS đề 2: 7
PASS đề 3: 4
PASS 1 từ hợp lệ: 1
PASS 1 từ không hợp lệ: 0
PASS đủ 26 chữ: 3
PASS ký tự lạ ở cuối: 0
PASS lặp ký tự: 1


In [23]:
all_funcs = [count_consistent_naive, count_consistent_set, count_consistent_subset,
             count_consistent_array, count_consistent_bitmask, count_consistent_complement,
             Solution().countConsistentStrings, SolutionBitmask().countConsistentStrings]

random.seed(2024)
for _ in range(500):
    k = random.randint(1, 26)
    al = "".join(random.sample(string.ascii_lowercase, k))
    ws = ["".join(random.choice("abcdefgh") for _ in range(random.randint(1, 10)))
          for _ in range(random.randint(1, 30))]
    results = {f(al, ws) for f in all_funcs}
    assert len(results) == 1, (al, ws, results)
print("Đối chiếu 8 cài đặt trên 500 bộ ngẫu nhiên: PASS")

Đối chiếu 8 cài đặt trên 500 bộ ngẫu nhiên: PASS


## 12. Lỗi người mới hay gặp

### Lỗi 1 — Đếm "có ký tự hợp lệ" thay vì "mọi ký tự hợp lệ"

Dùng `any` thay cho `all`, hoặc `count += 1` ngay khi thấy một ký tự hợp lệ.

In [24]:
def wrong_any(allowed, words):
    s = set(allowed)
    return sum(any(ch in s for ch in w) for w in words)   # SAI: chỉ cần 1 ký tự hợp lệ

print("sai:", wrong_any("ab", ["ad","bd","aaab","baa","badab"]), "| đúng:", count_consistent_set("ab", ["ad","bd","aaab","baa","badab"]))

sai: 5 | đúng: 2


**Sửa:** dùng `all(...)`, hoặc dùng cờ `ok` và chỉ cộng sau khi duyệt hết từ.

### Lỗi 2 — Cộng `count` bên trong vòng ký tự

`count` bị cộng nhiều lần cho cùng một từ.

In [25]:
def wrong_inner_count(allowed, words):
    count = 0
    for w in words:
        for ch in w:
            if ch in allowed:
                count += 1          # SAI: đang đếm ký tự, không phải đếm từ
    return count

print("sai:", wrong_inner_count("abc", ["ab", "abc"]), "| đúng:", count_consistent_set("abc", ["ab", "abc"]))

sai: 5 | đúng: 2


**Sửa:** đặt `count += 1` **sau** vòng ký tự, và chỉ khi cờ `ok` vẫn `True`.

### Lỗi 3 — Quên reset cờ `ok` cho mỗi từ

Khai báo `ok = True` ở ngoài vòng `for w` → một từ hỏng làm hỏng luôn mọi từ sau.

In [26]:
def wrong_flag(allowed, words):
    count = 0
    ok = True                       # SAI: chỉ khởi tạo 1 lần
    for w in words:
        for ch in w:
            if ch not in allowed:
                ok = False
                break
        if ok:
            count += 1
    return count

print("sai:", wrong_flag("ab", ["ad", "ab", "ba"]), "| đúng:", count_consistent_set("ab", ["ad", "ab", "ba"]))

sai: 0 | đúng: 2


**Sửa:** đưa `ok = True` vào **đầu** thân vòng `for w in words`.

### Lỗi 4 — Bitmask so sánh tập con **sai chiều**

Muốn hỏi "chữ của từ ⊆ allowed" nhưng lại viết `(wm & am) == am`, tức là hỏi "allowed ⊆ chữ của từ" — ngược hoàn toàn.

In [27]:
am = to_mask("ab")
for w in ["a", "ab", "abd"]:
    wm = to_mask(w)
    print(f"{w:>4}: sai chiều (wm & am) == am -> {str((wm & am) == am):<5} | đúng (wm & am) == wm -> {(wm & am) == wm}")

   a: sai chiều (wm & am) == am -> False | đúng (wm & am) == wm -> True
  ab: sai chiều (wm & am) == am -> True  | đúng (wm & am) == wm -> True
 abd: sai chiều (wm & am) == am -> True  | đúng (wm & am) == wm -> False


`"a"` consistent nhưng bị đánh giá sai thành `False`, còn `"abd"` không consistent lại bị đánh giá `True`.

**Sửa:** điều kiện đúng là `(wm & am) == wm` (mọi bit của từ đều có trong allowed), tương đương `(wm & ~am) == 0`. Lưu ý thêm: trong Python `&` ưu tiên **cao hơn** `==` (khác C/Java), nhưng vẫn nên đặt ngoặc cho dễ đọc và khi chuyển sang ngôn ngữ khác không bị lỗi.

## 13. Mẹo nhận dạng pattern

- Đề nói "**mọi** phần tử thuộc một tập cho phép / không chứa ký tự cấm" → `all(x in allowed_set ...)` hoặc kiểm tra tập con.
- Tập dữ liệu **cố định** bị hỏi "có/không" nhiều lần → tiền xử lý thành `set`, mảng đánh dấu hoặc bitmask.
- Bảng chữ cái nhỏ (26 chữ thường) → nghĩ đến **mảng 26 ô** hoặc **bitmask 26 bit**.
- "Đếm số phần tử thoả điều kiện" → `sum(cond for x in xs)`.
- Bài liên quan: 771 Jewels and Stones, 1832 Check if the Sentence Is Pangram, 2062/1941 (đếm ký tự), 318 Maximum Product of Word Lengths (bitmask chữ cái).

## 14. Biến thể

### Biến thể 1 — Trả về danh sách từ consistent (không chỉ đếm)

In [28]:
def list_consistent(allowed, words):
    s = set(allowed)
    return [w for w in words if all(ch in s for ch in w)]

print(list_consistent("cad", ["cc","acd","b","ba","bac","bad","ac","d"]))

['cc', 'acd', 'ac', 'd']


### Biến thể 2 — Danh sách **cấm** thay cho danh sách cho phép

Từ "sạch" nếu **không** chứa ký tự nào trong `banned`. Đây là phủ định: `not any(ch in banned_set for ch in w)`; với bitmask: `(wm & bm) == 0`.

In [29]:
def count_clean(banned, words):
    bm = to_mask(banned)
    return sum((to_mask(w) & bm) == 0 for w in words)

print(count_clean("xyz", ["abc", "axe", "hello", "zoo"]))   # abc, hello -> 2

2


### Biến thể 3 — Bitmask để kiểm tra hai từ không có chữ chung (ý tưởng LeetCode 318)

Hai từ không có chữ chung ⇔ `(mask1 & mask2) == 0`.

In [30]:
words_v = ["abcw", "baz", "foo", "bar", "xtfn", "abcdef"]
masks = [to_mask(w) for w in words_v]
best = 0
for i in range(len(words_v)):
    for j in range(i + 1, len(words_v)):
        if (masks[i] & masks[j]) == 0:
            best = max(best, len(words_v[i]) * len(words_v[j]))
print("tích độ dài lớn nhất của hai từ không chung chữ:", best)   # abcw * xtfn = 16

tích độ dài lớn nhất của hai từ không chung chữ: 16


## 15. Bài tập (tăng dần)

> Các hàm dưới đây chỉ có `pass` và **không được gọi** khi chạy notebook. Tự làm rồi bỏ comment các `assert`.

### Bài 1 — Dự đoán output (tính tay)
Không chạy code, hãy cho biết: `allowed = "xyz"`, `words = ["x", "xy", "xyzz", "a", "zx", "yyb"]` → đáp án bao nhiêu? Ghi lại từng từ ✓/✗.

In [31]:
def exercise_1_answer() -> int:
    # TODO: trả về con số bạn tính được bằng tay
    pass

# assert exercise_1_answer() == ...

### Bài 2 — Điền chỗ trống: hàm kiểm tra một từ
Viết `is_consistent(allowed_set, w)` dùng vòng `for` và `return` sớm (không dùng `all`).

In [32]:
def is_consistent(allowed_set: set, w: str) -> bool:
    # TODO 1: duyệt từng ký tự ch của w
    # TODO 2: nếu ch không thuộc allowed_set -> return False
    # TODO 3: duyệt xong -> return True
    pass

# assert is_consistent(set("ab"), "aaab") == True
# assert is_consistent(set("ab"), "abz") == False

### Bài 3 — Viết lại bằng mảng 26 ô
Viết `count_with_array(allowed, words)` không dùng `set`, không dùng `in` trên chuỗi.

In [33]:
def count_with_array(allowed: str, words: List[str]) -> int:
    # TODO: tạo ok = [False] * 26, bật ok cho chữ trong allowed, rồi đếm
    pass

# assert count_with_array("cad", ["cc","acd","b","ba","bac","bad","ac","d"]) == 4

### Bài 4 — Biến thể: đếm từ chứa **đủ mọi** chữ của `allowed`
Từ "đầy đủ" nếu mọi chữ của `allowed` đều xuất hiện trong từ (chiều ngược lại của tập con!).

In [34]:
def count_covering(allowed: str, words: List[str]) -> int:
    # TODO: allowed_set <= set(w)
    pass

# assert count_covering("ab", ["ab", "a", "bca", "bbb"]) == 2

### Bài 5 — Bitmask: nhóm từ theo tập chữ cái
Trả về dict `mask -> số từ có đúng tập chữ cái đó`, chỉ tính các từ consistent với `allowed`.

In [35]:
def group_by_mask(allowed: str, words: List[str]) -> dict:
    # TODO: tính am; với mỗi từ tính wm; nếu consistent thì tăng đếm cho wm
    pass

# r = group_by_mask("ab", ["a", "aa", "ab", "ba", "bd"])
# assert r == {1: 2, 3: 2}

### Bài 6 — Mini challenge: chữ cái hợp lệ ít nhất
Cho `words`, tìm chuỗi `allowed` (các chữ sắp tăng dần) **ngắn nhất** sao cho **mọi** từ đều consistent. Gợi ý: hợp (union) các tập.

In [36]:
def minimal_allowed(words: List[str]) -> str:
    # TODO: hợp tất cả set(w), rồi sắp xếp và nối lại
    pass

# assert minimal_allowed(["cab", "dd", "a"]) == "abcd"

## 16. Gợi ý 3 mức

**Bài 1:** (1) Tìm ký tự lạ trong từng từ. (2) Chữ `a` và `b` không thuộc `xyz`. (3) Chỉ `"a"` và `"yyb"` bị loại.

**Bài 2:** (1) Gặp ký tự lạ là biết kết quả ngay. (2) `for ch in w: if ch not in allowed_set: return False`. (3) Sau vòng lặp `return True`.

**Bài 3:** (1) Đổi chữ thành chỉ số 0..25. (2) `ord(ch) - ord('a')`. (3) `ok[ord(ch)-97]` là `False` → từ hỏng, `break`.

**Bài 4:** (1) Đổi chiều quan hệ tập con. (2) `set(allowed) <= set(w)`. (3) `sum(allowed_set <= set(w) for w in words)`.

**Bài 5:** (1) Dùng dict đếm. (2) `d[wm] = d.get(wm, 0) + 1`. (3) Điều kiện consistent: `(wm & ~am) == 0`.

**Bài 6:** (1) `allowed` phải chứa mọi chữ xuất hiện ở bất kỳ từ nào. (2) `u = set(); u |= set(w)`. (3) `"".join(sorted(u))`.

## 17. Đáp án tham khảo

Chỉ xem sau khi đã tự làm. Cell dưới định nghĩa đáp án và chạy kiểm tra.

In [37]:
def ans_exercise_1_answer() -> int:
    # x ✓, xy ✓, xyzz ✓, a ✗, zx ✓, yyb ✗
    return 4

def ans_is_consistent(allowed_set, w):
    for ch in w:
        if ch not in allowed_set:
            return False
    return True

def ans_count_with_array(allowed, words):
    ok = [False] * 26
    for ch in allowed:
        ok[ord(ch) - 97] = True
    count = 0
    for w in words:
        good = True
        for ch in w:
            if not ok[ord(ch) - 97]:
                good = False
                break
        if good:
            count += 1
    return count

def ans_count_covering(allowed, words):
    a = set(allowed)
    return sum(a <= set(w) for w in words)

def ans_group_by_mask(allowed, words):
    am = to_mask(allowed)
    d = {}
    for w in words:
        wm = to_mask(w)
        if (wm & ~am) == 0:
            d[wm] = d.get(wm, 0) + 1
    return d

def ans_minimal_allowed(words):
    u = set()
    for w in words:
        u |= set(w)
    return "".join(sorted(u))

assert ans_exercise_1_answer() == count_consistent_set("xyz", ["x", "xy", "xyzz", "a", "zx", "yyb"])
assert ans_is_consistent(set("ab"), "aaab") == True
assert ans_is_consistent(set("ab"), "abz") == False
assert ans_count_with_array("cad", ["cc","acd","b","ba","bac","bad","ac","d"]) == 4
assert ans_count_covering("ab", ["ab", "a", "bca", "bbb"]) == 2
assert ans_group_by_mask("ab", ["a", "aa", "ab", "ba", "bd"]) == {1: 2, 3: 2}
assert ans_minimal_allowed(["cab", "dd", "a"]) == "abcd"
w_all = ["cab", "dd", "a"]
assert count_consistent_set(ans_minimal_allowed(w_all), w_all) == len(w_all)
print("Tất cả đáp án PASS!")

Tất cả đáp án PASS!


## 18. Template nhớ nhanh + checklist submit

```python
class Solution:
    def countConsistentStrings(self, allowed: str, words: List[str]) -> int:
        allowed_set = set(allowed)                       # tiền xử lý 1 lần
        return sum(all(ch in allowed_set for ch in w)    # "mọi ký tự hợp lệ"
                   for w in words)                       # đếm bằng sum(bool)
```

**Khung tư duy:** *tập cho phép cố định* → *bảng tra nhanh* → *kiểm tra "với mọi"* → *đếm*.

**Checklist trước khi submit**
- [ ] Mình kiểm tra **mọi** ký tự (`all`), không phải "có một ký tự" (`any`)?
- [ ] Biến cờ/đếm được reset đúng chỗ cho từng từ?
- [ ] `count += 1` nằm **ngoài** vòng duyệt ký tự?
- [ ] Nếu dùng bitmask: điều kiện tập con đúng chiều `(wm & am) == wm`?
- [ ] Đã thử từ có ký tự lạ ở cuối và từ lặp ký tự?
- [ ] Đúng tên method `countConsistentStrings` và trả về `int`?